# 04.3 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/03-missing-and-outliers/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail()                                   # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

def customer_table(cutoff, months=3):
    """One row per customer: features from before `cutoff`, churned = no order in the next `months`."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=months))]
    c = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                        products=("StockCode", "nunique"), country=("Country", "first"),
                                        first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
    c["tenure_days"] = (cutoff - c["first_order"]).dt.days
    c["days_since_last"] = (cutoff - c["last_order"]).dt.days
    came_back = future.groupby("Customer ID")["Revenue"].sum().reindex(c.index).fillna(0) > 0
    c["churned"] = (~came_back).astype(int)
    return c.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
cust = customer_table("2011-09-01")
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

past = sales[sales["InvoiceDate"] < "2011-09-01"]
span = past.groupby("Customer ID")["InvoiceDate"].agg(["min", "max"])
days_active = (span["max"] - span["min"]).dt.days.reindex(cust.index)
# average days between orders; undefined (missing) for customers with only one order
cust["avg_gap_days"] = (days_active / (cust["orders"] - 1)).where(cust["orders"] > 1)
print(f"avg_gap_days missing for {cust['avg_gap_days'].isna().mean():.1%} of customers")

gap = pd.Series([12, 20, np.nan, 30, np.nan, 45, 90, np.nan], name="avg_gap_days")
country = pd.Series(["UK", "UK", "DE", np.nan, "UK", "FR", np.nan, "UK"], name="country")

spend8 = pd.Series([120, 150, 180, 200, 240, 260, 300, 4800], name="spend")

3,317 customers, churn rate 43.6%
avg_gap_days missing for 41.5% of customers


## Exercise 1

The training rows are 12, 20, ?, 30, ?. Known values: 12, 20, 30. Mean = 62 / 3 = **20.67**. All three gaps (train and test) get 20.67, not the 39.4 you would get using all eight rows.

In [2]:
from sklearn.impute import SimpleImputer
train, test = gap.iloc[:5].to_frame(), gap.iloc[5:].to_frame()
imp = SimpleImputer(strategy="mean").fit(train)
print("learned mean:", imp.statistics_[0].round(2))
print("train filled:", imp.transform(train).ravel().round(2))
print("test filled: ", imp.transform(test).ravel().round(2))

learned mean: 20.67
train filled: [12.   20.   20.67 30.   20.67]
test filled:  [45.   90.   20.67]


## Exercise 2

In [3]:
no_desc = raw[raw["Description"].isna()]
print(f"{len(no_desc):,} rows without a description")
print("price values:", no_desc["Price"].value_counts().head(3).to_dict())
print("customer id missing:", no_desc["Customer ID"].isna().mean().round(3))
print("quantity sign:", np.sign(no_desc["Quantity"]).value_counts().to_dict())

1,454 rows without a description
price values: {0.0: 1454}
customer id missing: 1.0
quantity sign: {-1: 862, 1: 592}


They have price 0 and no customer: warehouse adjustments (stock written off, damaged, found), not sales. The missingness is tied to *what kind of row it is*, so it is not random. They are not customer purchases, so the right treatment is to **remove them before modelling**, which the standard cleaning already does (price > 0 and a customer ID are required).

## Exercise 3

In [4]:
rest = spend8[spend8 < 4800]
m, s = rest.mean(), rest.std()
print(f"without the wholesaler: mean {m:.2f}, std {s:.2f}")
print(f"z of 4,800 against that = {(4800 - m) / s:.1f}")

without the wholesaler: mean 207.14, std 63.43
z of 4,800 against that = 72.4


Against the other seven customers, 4,800 is about 72 standard deviations away. The outlier was only "normal" when it was allowed to set the yardstick it was measured with. Robust alternatives use the median and the IQR (or the median absolute deviation) instead of the mean and standard deviation.

## Exercise 4

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
cols = ["days_since_last", "avg_gap_days"]
X_tr, X_te, y_tr, y_te = train_test_split(cust[cols], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
for c in [-1, 0, 999]:
    imp = SimpleImputer(strategy="constant", fill_value=c).fit(X_tr)
    t = DecisionTreeClassifier(max_depth=4, random_state=0).fit(imp.transform(X_tr), y_tr)
    print(f"fill {c:>4}: test accuracy {t.score(imp.transform(X_te), y_te):.1%}")

fill   -1: test accuracy 67.5%
fill    0: test accuracy 67.5%
fill  999: test accuracy 66.7%


−1 and 0 give 67.5%, 999 gives 66.7%: all close, and all better than the logistic regression in the lesson. (The small difference comes from where the tree places its thresholds when the sentinel sits below or above the real gaps.) A tree can put an "impossible" value like −1 or 999 in its own branch, so for trees a **constant outside the real range** works like a built-in missing indicator. For linear models a constant like 999 is harmful: it is treated as a real, huge gap. That is why linear models want median + indicator, and trees are happy with a sentinel constant.